# Ch.0 Prerequisites Demo: Numeric Walkthroughs

This notebook implements the worked examples from Ch.0 so you can experiment with the numbers and build intuition.

**Sections:**
1. Embeddings and dot product similarity
2. Vanishing gradient exponential decay
3. Complete 3-token attention mechanism
4. Skip connection gradient flow
5. Cross-entropy loss calculation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=3, suppress=True)

## 1. Embeddings: Why Text Becomes Vectors

From Ch.0 §1: Embeddings map tokens to vectors where **distance = similarity**.

In [ ]:
# Example embeddings (3-dimensional for visualization)
embeddings = {
    "cat": np.array([0.8, 0.9, 0.1]),
    "dog": np.array([0.7, 0.8, 0.2]),
    "car": np.array([0.1, 0.2, 0.9])
}

def dot_product_similarity(word1, word2):
    """Compute dot product between two word embeddings."""
    v1 = embeddings[word1]
    v2 = embeddings[word2]
    similarity = np.dot(v1, v2)

    print(f"{word1} · {word2} = {v1[0]:.1f}×{v2[0]:.1f} + {v1[1]:.1f}×{v2[1]:.1f} + {v1[2]:.1f}×{v2[2]:.1f}")
    print(f"         = {v1[0]*v2[0]:.2f} + {v1[1]*v2[1]:.2f} + {v1[2]*v2[2]:.2f}")
    print(f"         = {similarity:.2f}")
    return similarity

print("Dot product similarity (higher = more similar):")
print()
sim_cat_dog = dot_product_similarity("cat", "dog")
print()
sim_cat_car = dot_product_similarity("cat", "car")
print()
print(f"Interpretation: 'cat' and 'dog' are {sim_cat_dog/sim_cat_car:.1f}× more similar than 'cat' and 'car'")

In [ ]:
# Visualize embeddings in 3D space
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for word, vec in embeddings.items():
    ax.scatter(vec[0], vec[1], vec[2], s=200, label=word)
    ax.text(vec[0], vec[1], vec[2], f'  {word}', fontsize=12)

ax.set_xlabel('Dimension 1')
ax.set_ylabel('Dimension 2')
ax.set_zlabel('Dimension 3')
ax.set_title('Word Embeddings in 3D Space\n(Closer points = more similar meanings)')
ax.legend()
plt.tight_layout()
plt.show()

print("Notice: 'cat' and 'dog' are close together (both animals)")
print("        'car' is far away (different semantic category)")

## 2. Vanishing Gradient Problem

From Ch.0 §2.2: Gradients decay exponentially when backpropagating through RNN time steps.

In [ ]:
# Simulate gradient decay through T time steps
decay_factor = 0.8  # Typical for tanh activation
time_steps = np.array([1, 5, 10, 20, 50, 100])

gradients = decay_factor ** time_steps
percentages = gradients * 100

print("Gradient magnitude after T time steps (decay_factor = 0.8):")
print()
print("Steps (T) | Gradient Magnitude | % of Original")
print("-" * 50)
for t, grad, pct in zip(time_steps, gradients, percentages):
    print(f"{t:8d}  | {grad:18.6f} | {pct:12.4f}%")

print()
print(f"After 100 tokens, gradient is {percentages[-1]:.6f}% of original")
print("→ Early tokens effectively cannot learn!")

In [ ]:
# Visualize exponential decay
t_range = np.arange(1, 101)
gradients_range = decay_factor ** t_range

plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(t_range, gradients_range, linewidth=2, color='red')
plt.xlabel('Time Steps (T)', fontsize=12)
plt.ylabel('Gradient Magnitude', fontsize=12)
plt.title('Vanishing Gradient Through Time', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=0.01, color='gray', linestyle='--', label='1% threshold')
plt.legend()

plt.subplot(1, 2, 2)
plt.semilogy(t_range, gradients_range, linewidth=2, color='red')
plt.xlabel('Time Steps (T)', fontsize=12)
plt.ylabel('Gradient Magnitude (log scale)', fontsize=12)
plt.title('Exponential Decay (Log Scale)', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Left plot: Linear scale shows gradients approaching zero")
print("Right plot: Log scale reveals exponential decay rate")

## 3. Attention Mechanism: Complete 3-Token Walkthrough

From Ch.0 §3.2: Implementing the exact "The river bank" example with all calculations shown.

In [ ]:
# Sentence: "The river bank"
# 2-D embeddings for simplicity
X = np.array([
    [1.0, 0.2],  # "The"
    [0.8, 0.9],  # "river"
    [0.5, 1.1]   # "bank"
])

print("Input embeddings X (3 tokens × 2 dimensions):")
print(X)
print()

# For simplicity, use identity projection (Q = K = V = X)
Q = K = V = X

print("Step 1: Compute attention scores (Q·K^T)")
scores = Q @ K.T
print("Scores matrix:")
print(scores)
print()

# Show detailed calculation for one query
print("Detailed calculation for 'bank' (row 3):")
q_bank = Q[2]
print(f"  q_bank = {q_bank}")
print()
for i, word in enumerate(["The", "river", "bank"]):
    k_i = K[i]
    score = np.dot(q_bank, k_i)
    print(f"  score(bank, {word}) = [{q_bank[0]:.1f}, {q_bank[1]:.1f}] · [{k_i[0]:.1f}, {k_i[1]:.1f}]")
    print(f"                       = {q_bank[0]:.1f}×{k_i[0]:.1f} + {q_bank[1]:.1f}×{k_i[1]:.1f}")
    print(f"                       = {score:.2f}")
    print()

In [ ]:
print("Step 2: Apply softmax (row-wise normalization)")
print()

def softmax(x):
    """Compute softmax values for vector x."""
    exp_x = np.exp(x)
    return exp_x / exp_x.sum()

attention_weights = np.zeros_like(scores)
for i in range(len(scores)):
    attention_weights[i] = softmax(scores[i])

# Show detailed calculation for 'bank' row
scores_bank = scores[2]
print(f"Scores for 'bank': {scores_bank}")
print()
exp_scores = np.exp(scores_bank)
print(f"Exponentiate: exp({scores_bank[0]:.2f}) = {exp_scores[0]:.2f}")
print(f"              exp({scores_bank[1]:.2f}) = {exp_scores[1]:.2f}")
print(f"              exp({scores_bank[2]:.2f}) = {exp_scores[2]:.2f}")
print()
sum_exp = exp_scores.sum()
print(f"Sum: {exp_scores[0]:.2f} + {exp_scores[1]:.2f} + {exp_scores[2]:.2f} = {sum_exp:.2f}")
print()
alpha_bank = attention_weights[2]
print(f"Normalize: α_bank = [{exp_scores[0]:.2f}/{sum_exp:.2f}, {exp_scores[1]:.2f}/{sum_exp:.2f}, {exp_scores[2]:.2f}/{sum_exp:.2f}]")
print(f"                  = {alpha_bank}")
print()
print("Full attention weight matrix:")
print(attention_weights)
print()
print("Interpretation: Each row sums to 1.0 (probability distribution)")

In [ ]:
print("Step 3: Weighted sum of values")
print()

output = attention_weights @ V

# Show detailed calculation for 'bank'
alpha_bank = attention_weights[2]
print(f"Output for 'bank':")
print(f"  = {alpha_bank[0]:.3f} × [1.0, 0.2] + {alpha_bank[1]:.3f} × [0.8, 0.9] + {alpha_bank[2]:.3f} × [0.5, 1.1]")
print(f"  = [{alpha_bank[0]*V[0,0]:.3f}, {alpha_bank[0]*V[0,1]:.3f}] + [{alpha_bank[1]*V[1,0]:.3f}, {alpha_bank[1]*V[1,1]:.3f}] + [{alpha_bank[2]*V[2,0]:.3f}, {alpha_bank[2]*V[2,1]:.3f}]")
print(f"  = {output[2]}")
print()
print("Full output matrix:")
print(output)
print()
print(f"'bank' attended {alpha_bank[2]*100:.1f}% to itself and {alpha_bank[1]*100:.1f}% to 'river'")
print("→ Context from 'river' disambiguates 'bank' as geographic feature")

In [ ]:
# Visualize attention weights as heatmap
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.heatmap(attention_weights,
            annot=True,
            fmt='.3f',
            cmap='YlOrRd',
            xticklabels=['The', 'river', 'bank'],
            yticklabels=['The', 'river', 'bank'],
            cbar_kws={'label': 'Attention Weight'})
plt.title('Attention Weight Matrix\n(Row = Query, Column = Key)', fontsize=14)
plt.xlabel('Attending TO', fontsize=12)
plt.ylabel('Attending FROM', fontsize=12)
plt.tight_layout()
plt.show()

print("Each row shows how much that token attends to others")
print("Diagonal: Self-attention weights")
print("Off-diagonal: Cross-token attention")

### Why Scaled Dot-Product? (√d_k factor)

In [ ]:
# Demonstrate softmax saturation without scaling
d_k = 64  # Typical transformer dimension

# Simulate scores for d_k=64 (magnitudes scale with dimension)
large_scores = np.array([82.0, 15.0, -38.0])
scaled_scores = large_scores / np.sqrt(d_k)

print(f"With d_k={d_k} dimensions:")
print()
print("UNSCALED scores:", large_scores)
alpha_unscaled = softmax(large_scores)
print("Softmax output:", alpha_unscaled)
print(f"→ Collapsed to near one-hot: {alpha_unscaled[0]*100:.2f}% on first token")
print()
print("SCALED scores (÷ √64 = 8):", scaled_scores)
alpha_scaled = softmax(scaled_scores)
print("Softmax output:", alpha_scaled)
print(f"→ Balanced distribution: {alpha_scaled[0]*100:.1f}%, {alpha_scaled[1]*100:.1f}%, {alpha_scaled[2]*100:.1f}%")
print()
print("Scaling prevents saturation → better gradient flow!")

## 4. Skip Connections: The "+1" Gradient Highway

From Ch.0 §4: Residual connections preserve gradient flow through deep networks.

In [ ]:
# Simulate gradient backpropagation through 40 layers
num_layers = 40
initial_gradient = 1.0

# Plain network: gradient multiplied by weight derivative at each layer
# Assume average derivative is 0.9 per layer
plain_decay = 0.9
gradients_plain = [initial_gradient]
for layer in range(num_layers):
    gradients_plain.append(gradients_plain[-1] * plain_decay)

# ResNet: gradient = upstream × (dF/dx + 1)
# Even if dF/dx ≈ 0, the "+1" preserves flow
resnet_decay = 0.95  # Higher because of "+1" term
gradients_resnet = [initial_gradient]
for layer in range(num_layers):
    gradients_resnet.append(gradients_resnet[-1] * resnet_decay)

print(f"Gradient magnitude after {num_layers} layers:")
print()
print(f"Plain network: {gradients_plain[-1]:.6f} ({gradients_plain[-1]*100:.2f}% of original)")
print(f"ResNet:        {gradients_resnet[-1]:.6f} ({gradients_resnet[-1]*100:.2f}% of original)")
print()
print(f"ResNet preserves {(gradients_resnet[-1]/gradients_plain[-1]):.1f}× more gradient!")

In [ ]:
# Visualize gradient flow comparison
layers = np.arange(0, num_layers + 1)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(layers, gradients_plain, label='Plain Network', linewidth=2, color='red')
plt.plot(layers, gradients_resnet, label='ResNet (with skip connections)', linewidth=2, color='green')
plt.xlabel('Layer Depth', fontsize=12)
plt.ylabel('Gradient Magnitude', fontsize=12)
plt.title('Gradient Flow: Plain vs ResNet', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axhline(y=0.1, color='gray', linestyle='--', alpha=0.5, label='10% threshold')

plt.subplot(1, 2, 2)
plt.semilogy(layers, gradients_plain, label='Plain Network', linewidth=2, color='red')
plt.semilogy(layers, gradients_resnet, label='ResNet', linewidth=2, color='green')
plt.xlabel('Layer Depth', fontsize=12)
plt.ylabel('Gradient Magnitude (log scale)', fontsize=12)
plt.title('Log Scale Comparison', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The '+1' term in dL/dx = dL/dy × (dF/dx + 1) creates a gradient highway")
print("This is why 96-layer GPT-3 can train successfully!")

## 5. Cross-Entropy Loss

From Ch.0 §6.2: Measuring prediction quality for next-token prediction.

In [ ]:
# Example: Predicting "cat" after "The"
# Model outputs probability distribution over vocabulary

def cross_entropy_loss(true_token_prob):
    """Calculate cross-entropy loss for a single prediction."""
    return -np.log(true_token_prob)

print("Predicting next word after 'The':")
print()

# Scenario 1: Poor model
print("Scenario 1: Poorly trained model")
probs_poor = {'cat': 0.3, 'dog': 0.4, 'car': 0.1, 'house': 0.2}
true_word = 'cat'
print(f"  Model predictions: {probs_poor}")
print(f"  True next word: '{true_word}'")
loss_poor = cross_entropy_loss(probs_poor[true_word])
print(f"  Loss = -log({probs_poor[true_word]}) = {loss_poor:.3f}")
print()

# Scenario 2: Better model
print("Scenario 2: Better trained model")
probs_better = {'cat': 0.8, 'dog': 0.1, 'car': 0.05, 'house': 0.05}
print(f"  Model predictions: {probs_better}")
print(f"  True next word: '{true_word}'")
loss_better = cross_entropy_loss(probs_better[true_word])
print(f"  Loss = -log({probs_better[true_word]}) = {loss_better:.3f}")
print()

print(f"Improvement: Loss decreased from {loss_poor:.3f} → {loss_better:.3f}")
print(f"Lower loss = better predictions!")

In [ ]:
# Visualize relationship between probability and loss
probabilities = np.linspace(0.01, 1.0, 100)
losses = -np.log(probabilities)

plt.figure(figsize=(10, 6))
plt.plot(probabilities, losses, linewidth=2, color='blue')
plt.scatter([probs_poor[true_word]], [loss_poor], s=200, color='red',
            label=f'Poor model (P={probs_poor[true_word]}, Loss={loss_poor:.2f})', zorder=5)
plt.scatter([probs_better[true_word]], [loss_better], s=200, color='green',
            label=f'Better model (P={probs_better[true_word]}, Loss={loss_better:.2f})', zorder=5)
plt.xlabel('Probability of True Token', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.title('Cross-Entropy Loss Function', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print("Key insight: As model confidence in correct token increases,")
print("             loss decreases (training is working!)")

## Summary: You're Ready for Ch.1!

You've now seen concrete implementations of:

✅ **Embeddings** — Vectors encode semantic similarity via dot products  
✅ **Vanishing gradients** — Exponential decay (0.8^T) prevents deep RNN learning  
✅ **Attention mechanism** — Complete Q/K/V walkthrough with 3 tokens  
✅ **Skip connections** — "+1" gradient term preserves flow through 40+ layers  
✅ **Cross-entropy loss** — How models measure prediction quality  

**Next:** Ch.1 adds tokenization (BPE), multi-head attention (parallel Q/K/V), positional encoding, and the full transformer architecture.

These foundations transfer directly—you're now equipped to understand how BERT, GPT, and modern LLMs work!